# Return-Preserving Hybrid Failure Repair Screen v1

This notebook is a diagnostic target-repair screen for `return_preserving_hybrid_target_bc_replay_v1`. It does **not** promote an actor.

The screen tests whether the selected hybrid official daily paths can be repaired by a simple path-level combination of:

- stress caps to keep `wf_2022` MaxDD below 20%;
- bull participation floors to restore `wf_2023_2024` relative return and `wf_2020` return floor.

The replay is an official-return-scaled position diagnostic: it reproduces the selected official paths before overlays, then scales daily strategy returns by the candidate applied exposure. Passing this notebook would only create a target/replay handoff for a new BC/checkpoint run; it would not be retained model evidence by itself.

In [1]:
from __future__ import annotations

import itertools
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "SPEC.md").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError(f"Could not locate repo root from {start}")


ROOT = find_repo_root(Path.cwd())
EXPERIMENT = "return_preserving_hybrid_failure_repair_screen_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime/model_selection/card/return_preserving_hybrid_target_bc_replay_v1"
CONTRACT_CARD = ROOT / "runtime/model_selection/card/cap10_wf2020_return_preserving_target_revision_v1"
CARD.mkdir(parents=True, exist_ok=True)


def rel(path: Path) -> str:
    return str(Path(path).resolve().relative_to(ROOT))


FOLDS = ["wf_2020", "wf_2022", "wf_2023_2024"]
TARGET_RELATIVE_PP = {"wf_2020": -263.7287958891489, "wf_2022": 30.0, "wf_2023_2024": 30.0}
RETURN_FLOOR_PCT = {"wf_2020": 69.60455156850816, "wf_2022": None, "wf_2023_2024": None}
TARGET_MAX_DD_PCT = 20.0
SELECTED_CANDIDATE = "hybrid_blend_nonrisk_50__riskcap_default_weight"

source_summary = json.loads((SOURCE_CARD / "summary.json").read_text())
contract = json.loads((CONTRACT_CARD / "target_revision_contract.json").read_text())

print("repo root:", ROOT)
print("source decision:", source_summary["decision"])
print("selected candidate:", source_summary["selected_candidate"])
print("next mechanism:", source_summary["next_mechanism"])


repo root: /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
source decision: return_preserving_hybrid_bc_replay_no_promotion
selected candidate: hybrid_blend_nonrisk_50__riskcap_default_weight
next mechanism: inspect_selected_hybrid_failure_and_repair_participation_weight_or_target_before_retention_training


## Replay Helpers

`return_preserving_hybrid_target_bc_replay_v1` stores official daily paths, not the original policy logits. This screen therefore uses a path-level replay approximation, but with a strict baseline check: the selected path must be reproduced before any overlay.

The applied exposure rule is previous-day position, except when previous-day position is zero and the current row opens a position; that BUY-row special case is needed because the official path applies some entry exposure on the same date. This rule reproduces all three selected official paths to floating-point error.

In [2]:
def _clean(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): _clean(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_clean(v) for v in value]
    if isinstance(value, tuple):
        return [_clean(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        if np.isnan(value) or np.isinf(value):
            return None
        return float(value)
    return value


def official_metrics(df: pd.DataFrame) -> dict[str, float]:
    nav = df["strategy_nav"].astype(float)
    buyhold = df["buyhold_nav"].astype(float)
    return_pct = (nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0
    buyhold_return_pct = (buyhold.iloc[-1] / buyhold.iloc[0] - 1.0) * 100.0
    max_dd_pct = (1.0 - nav / nav.cummax()).max() * 100.0
    return {
        "return_pct": float(return_pct),
        "buyhold_return_pct": float(buyhold_return_pct),
        "relative_return_pp": float(return_pct - buyhold_return_pct),
        "max_dd_pct": float(max_dd_pct),
        "avg_pos": float(df["position"].mean()),
        "trade_count": int((df["position"].diff().abs().fillna(0) > 0.02).sum()),
    }


def applied_exposure(position: np.ndarray) -> np.ndarray:
    previous = np.r_[position[0], position[:-1]]
    return np.where((previous < 1e-9) & (position > 0), position, previous)


def load_fold(fold: str) -> dict[str, Any]:
    path = SOURCE_CARD / f"{SELECTED_CANDIDATE}_{fold}_official_daily_path.csv"
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    price = df["price"].astype(float)
    nav = df["strategy_nav"].astype(float)
    position = df["position"].astype(float)

    mom21_prev = price.pct_change(21).shift(1).fillna(0.0).to_numpy(float)
    ma50 = price.rolling(50, min_periods=10).mean()
    dist_ma50_prev = (price / ma50 - 1.0).shift(1).fillna(0.0).to_numpy(float)
    high63 = price.rolling(63, min_periods=10).max()
    px_dd63_prev = (price / high63 - 1.0).shift(1).fillna(0.0).to_numpy(float)
    base_dd_prev = (nav / nav.cummax() - 1.0).shift(1).fillna(0.0).to_numpy(float)

    base_applied = pd.Series(applied_exposure(position.to_numpy(float)), index=df.index)
    unit_return = (
        nav.pct_change().fillna(0.0)
        / base_applied.replace(0.0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(price.pct_change().fillna(0.0)).to_numpy(float)

    return {
        "fold": fold,
        "path": rel(path),
        "df": df,
        "position": position.to_numpy(float),
        "unit_return": unit_return,
        "nav_start": float(nav.iloc[0]),
        "buyhold_start": float(df["buyhold_nav"].iloc[0]),
        "buyhold_end": float(df["buyhold_nav"].iloc[-1]),
        "mom21_prev": mom21_prev,
        "dist_ma50_prev": dist_ma50_prev,
        "px_dd63_prev": px_dd63_prev,
        "base_dd_prev": base_dd_prev,
        "official_metrics": official_metrics(df),
    }


def replay_metrics(fold_data: dict[str, Any], position: np.ndarray) -> dict[str, Any]:
    position = np.clip(np.asarray(position, dtype=float), 0.0, 1.0)
    exposure = applied_exposure(position)
    daily = 1.0 + exposure * fold_data["unit_return"]
    daily[0] = 1.0
    nav = fold_data["nav_start"] * np.cumprod(daily)
    buyhold_return_pct = (fold_data["buyhold_end"] / fold_data["buyhold_start"] - 1.0) * 100.0
    return_pct = (nav[-1] / nav[0] - 1.0) * 100.0
    max_dd_pct = (1.0 - nav / np.maximum.accumulate(nav)).max() * 100.0
    return {
        "return_pct": float(return_pct),
        "buyhold_return_pct": float(buyhold_return_pct),
        "relative_return_pp": float(return_pct - buyhold_return_pct),
        "max_dd_pct": float(max_dd_pct),
        "avg_pos": float(position.mean()),
        "trade_count": int(np.sum(np.abs(np.diff(position, prepend=position[0])) > 0.02)),
        "end_nav": float(nav[-1]),
        "nav_path": nav,
    }


def make_position(
    fold_data: dict[str, Any],
    *,
    scale: float,
    global_cap: float,
    stress_cap: float,
    stress_dd: float,
    stress_mom21: float,
    bull_min_pos: float,
    bull_mom21: float,
    bull_dd63_floor: float,
) -> np.ndarray:
    position = np.minimum(fold_data["position"] * scale, global_cap)
    bull_mask = (
        (fold_data["dist_ma50_prev"] > 0.0)
        & (fold_data["mom21_prev"] >= bull_mom21)
        & (fold_data["px_dd63_prev"] >= bull_dd63_floor)
    )
    position = np.where(bull_mask, np.maximum(position, bull_min_pos), position)
    stress_mask = (
        (fold_data["base_dd_prev"] <= stress_dd)
        | ((fold_data["dist_ma50_prev"] < 0.0) & (fold_data["mom21_prev"] <= stress_mom21))
    )
    position = np.where(stress_mask, np.minimum(position, stress_cap), position)
    return np.clip(position, 0.0, 1.0)


def gate_metrics(fold: str, metrics: dict[str, Any]) -> dict[str, Any]:
    relative_gap = metrics["relative_return_pp"] - TARGET_RELATIVE_PP[fold]
    max_dd_gap = TARGET_MAX_DD_PCT - metrics["max_dd_pct"]
    floor = RETURN_FLOOR_PCT[fold]
    return_floor_gap = None if floor is None else metrics["return_pct"] - floor
    relative_gate = relative_gap >= 0.0
    max_dd_gate = max_dd_gap > 0.0
    return_floor_gate = True if return_floor_gap is None else return_floor_gap >= 0.0
    return {
        "relative_gap_pp": float(relative_gap),
        "max_dd_gap_pct": float(max_dd_gap),
        "return_floor_gap_pp": None if return_floor_gap is None else float(return_floor_gap),
        "relative_gate": bool(relative_gate),
        "max_dd_gate": bool(max_dd_gate),
        "return_floor_gate": bool(return_floor_gate),
        "fold_gate": bool(relative_gate and max_dd_gate and return_floor_gate),
    }

In [3]:
fold_data = {fold: load_fold(fold) for fold in FOLDS}

baseline_rows = []
for fold, fd in fold_data.items():
    replay = replay_metrics(fd, fd["position"])
    official = fd["official_metrics"]
    nav_error = float(np.max(np.abs(replay["nav_path"] - fd["df"]["strategy_nav"].to_numpy(float))))
    row = {"fold": fold, "nav_max_abs_error": nav_error}
    row.update({f"official_{k}": v for k, v in official.items()})
    row.update({f"replay_{k}": v for k, v in replay.items() if k not in {"nav_path", "end_nav"}})
    baseline_rows.append(row)

baseline_check = pd.DataFrame(baseline_rows)
baseline_check.to_csv(CARD / "baseline_replay_check.csv", index=False)
baseline_check

,fold,nav_max_abs_error,official_return_pct,official_buyhold_return_pct,official_relative_return_pp,official_max_dd_pct,official_avg_pos,official_trade_count,replay_return_pct,replay_buyhold_return_pct,replay_relative_return_pp,replay_max_dd_pct,replay_avg_pos,replay_trade_count
0,wf_2020,9.313226e-10,12.786482,58.093571,-45.307089,14.833673,0.181826,36,12.786482,58.093571,-45.307089,14.833673,0.181826,36
1,wf_2022,3.492460e-10,-21.049455,-86.588072,65.538617,21.744476,0.135038,39,-21.049455,-86.588072,65.538617,21.744476,0.135038,39
2,wf_2023_2024,2.561137e-09,14.025388,-0.356635,14.382023,14.651499,0.167957,33,14.025388,-0.356635,14.382023,14.651499,0.167957,33


## Grid Screen

The grid deliberately tests only mechanisms that could become target-contract changes:

- `stress_cap`, `stress_dd`, `stress_mom21`: lower exposure after visible drawdown or downtrend stress.
- `bull_min_pos`, `bull_mom21`, `bull_dd63_floor`: require minimum exposure during visible uptrends that are not too far below the recent high.
- `scale` and `global_cap`: small global exposure adjustments, still capped.

The grid has enough high-participation variants to rule out a trivial parameter-limit miss, but it stores only ranked summaries to avoid committing a huge all-combination CSV.

In [4]:
GRID = {
    "scale": [1.0, 1.25, 1.5, 2.0, 2.5, 3.0],
    "global_cap": [0.55, 0.65, 0.80, 1.00],
    "stress_cap": [0.04, 0.06, 0.08, 0.10, 0.12, 0.15],
    "stress_dd": [-0.06, -0.08, -0.10, -0.12, -0.15],
    "stress_mom21": [-0.01, -0.03, -0.05, -0.08],
    "bull_min_pos": [0.45, 0.55, 0.65, 0.75, 0.85, 1.00],
    "bull_mom21": [-0.02, 0.00, 0.02, 0.04, 0.06, 0.08, 0.10],
    "bull_dd63_floor": [-0.20, -0.15, -0.12, -0.08],
}

def evaluate_variant(params: dict[str, float]) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    fold_rows = []
    gaps = []
    for fold, fd in fold_data.items():
        position = make_position(fd, **params)
        metrics = replay_metrics(fd, position)
        gates = gate_metrics(fold, metrics)
        floor_gap = 999.0 if gates["return_floor_gap_pp"] is None else gates["return_floor_gap_pp"]
        gaps.extend([gates["relative_gap_pp"], gates["max_dd_gap_pct"], floor_gap])
        fold_rows.append({
            "variant": variant_name(params),
            "fold": fold,
            **params,
            **{k: v for k, v in metrics.items() if k != "nav_path"},
            **gates,
        })
    row = {
        "variant": variant_name(params),
        **params,
        "fold_gate_pass_count": sum(row["fold_gate"] for row in fold_rows),
        "max_dd_pass_count": sum(row["max_dd_gate"] for row in fold_rows),
        "relative_pass_count": sum(row["relative_gate"] for row in fold_rows),
        "return_floor_pass_count": sum(row["return_floor_gate"] for row in fold_rows),
        "min_gate_gap": float(min(gaps)),
        "mean_return_pct": float(np.mean([row["return_pct"] for row in fold_rows])),
        "worst_max_dd_pct": float(max(row["max_dd_pct"] for row in fold_rows)),
        "min_relative_return_pp": float(min(row["relative_return_pp"] for row in fold_rows)),
        "mean_avg_pos": float(np.mean([row["avg_pos"] for row in fold_rows])),
        "total_trade_count": int(sum(row["trade_count"] for row in fold_rows)),
    }
    return row, fold_rows


def variant_name(params: dict[str, float]) -> str:
    return (
        f"scale{params['scale']:.2f}_cap{params['global_cap']:.2f}"
        f"_stress{params['stress_cap']:.2f}_dd{abs(params['stress_dd']):.2f}_mom{params['stress_mom21']:.2f}"
        f"_bull{params['bull_min_pos']:.2f}_mom{params['bull_mom21']:.2f}_dd{abs(params['bull_dd63_floor']):.2f}"
    )

rows = []
fold_rows_by_variant: dict[str, list[dict[str, Any]]] = {}
candidate_count = 0
for values in itertools.product(*(GRID[key] for key in GRID)):
    params = dict(zip(GRID.keys(), values))
    if params["bull_min_pos"] > params["global_cap"]:
        continue
    summary_row, fold_rows = evaluate_variant(params)
    rows.append(summary_row)
    fold_rows_by_variant[summary_row["variant"]] = fold_rows
    candidate_count += 1

rank_cols = [
    "fold_gate_pass_count",
    "max_dd_pass_count",
    "relative_pass_count",
    "return_floor_pass_count",
    "min_gate_gap",
    "mean_return_pct",
]
grid_summary = pd.DataFrame(rows).sort_values(rank_cols, ascending=[False, False, False, False, False, False]).reset_index(drop=True)

# Persist ranked evidence without committing the full 300k-row grid. The notebook is the reproducible full grid definition.
grid_summary.head(5000).to_csv(CARD / "repair_grid_top.csv", index=False)

selected_variants = grid_summary.head(100)["variant"].tolist()
fold_repair_metrics = pd.DataFrame([row for variant in selected_variants for row in fold_rows_by_variant[variant]])
fold_repair_metrics.to_csv(CARD / "fold_repair_metrics.csv", index=False)

pass_distribution = (
    grid_summary.groupby(["fold_gate_pass_count", "max_dd_pass_count", "relative_pass_count", "return_floor_pass_count"], dropna=False)
    .size()
    .reset_index(name="candidate_count")
    .sort_values(["fold_gate_pass_count", "max_dd_pass_count", "relative_pass_count", "return_floor_pass_count"], ascending=False)
)
pass_distribution.to_csv(CARD / "pass_distribution.csv", index=False)

print("candidate_count", candidate_count)
print("strict_gate_pass_count", int((grid_summary["fold_gate_pass_count"] == len(FOLDS)).sum()))
grid_summary.head(20)

candidate_count 302400
strict_gate_pass_count 0


,variant,scale,global_cap,stress_cap,stress_dd,stress_mom21,bull_min_pos,bull_mom21,bull_dd63_floor,fold_gate_pass_count,max_dd_pass_count,relative_pass_count,return_floor_pass_count,min_gate_gap,mean_return_pct,worst_max_dd_pct,min_relative_return_pp,mean_avg_pos,total_trade_count
0,scale1.00_cap0.55_stress0.15_dd0.06_mom-0.01_b...,1.0,0.55,0.15,-0.06,-0.01,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.929363,18.760305,-11.177543,0.210146,86
1,scale1.00_cap0.55_stress0.15_dd0.06_mom-0.03_b...,1.0,0.55,0.15,-0.06,-0.03,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.929363,18.760305,-11.177543,0.210146,86
2,scale1.00_cap0.65_stress0.15_dd0.06_mom-0.01_b...,1.0,0.65,0.15,-0.06,-0.01,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.929363,18.760305,-11.177543,0.210146,86
3,scale1.00_cap0.65_stress0.15_dd0.06_mom-0.03_b...,1.0,0.65,0.15,-0.06,-0.03,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.929363,18.760305,-11.177543,0.210146,86
4,scale1.00_cap0.80_stress0.15_dd0.06_mom-0.01_b...,1.0,0.80,0.15,-0.06,-0.01,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.929363,18.760305,-11.177543,0.210146,86
5,scale1.00_cap0.80_stress0.15_dd0.06_mom-0.03_b...,1.0,0.80,0.15,-0.06,-0.03,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.929363,18.760305,-11.177543,0.210146,86
6,scale1.00_cap1.00_stress0.15_dd0.06_mom-0.01_b...,1.0,1.00,0.15,-0.06,-0.01,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.929363,18.760305,-11.177543,0.210146,86
7,scale1.00_cap1.00_stress0.15_dd0.06_mom-0.03_b...,1.0,1.00,0.15,-0.06,-0.03,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.929363,18.760305,-11.177543,0.210146,86
8,scale1.00_cap0.55_stress0.15_dd0.06_mom-0.05_b...,1.0,0.55,0.15,-0.06,-0.05,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.902846,18.836177,-11.177543,0.210157,86
9,scale1.00_cap0.65_stress0.15_dd0.06_mom-0.05_b...,1.0,0.65,0.15,-0.06,-0.05,0.55,0.1,-0.2,2,3,3,2,-22.688524,21.902846,18.836177,-11.177543,0.210157,86


In [5]:
best = grid_summary.iloc[0].to_dict()
best_fold_rows = pd.DataFrame(fold_rows_by_variant[best["variant"]]).sort_values("fold")

# Useful alternates for diagnosis: best drawdown-safe and best wf2020 return-floor candidate if they differ.
drawdown_safe = grid_summary[grid_summary["max_dd_pass_count"] == len(FOLDS)].copy()
best_drawdown_safe = drawdown_safe.iloc[0].to_dict() if not drawdown_safe.empty else None

return_floor_safe = grid_summary[grid_summary["return_floor_pass_count"] == len(FOLDS)].copy()
best_return_floor_safe = return_floor_safe.iloc[0].to_dict() if not return_floor_safe.empty else None

strict_pass = grid_summary[grid_summary["fold_gate_pass_count"] == len(FOLDS)].copy()
strict_gate_pass_count = int(len(strict_pass))

best_fold_rows

,variant,fold,scale,global_cap,stress_cap,stress_dd,stress_mom21,bull_min_pos,bull_mom21,bull_dd63_floor,...,avg_pos,trade_count,end_nav,relative_gap_pp,max_dd_gap_pct,return_floor_gap_pp,relative_gate,max_dd_gate,return_floor_gate,fold_gate
0,scale1.00_cap0.55_stress0.15_dd0.06_mom-0.01_b...,wf_2020,1.0,0.55,0.15,-0.06,-0.01,0.55,0.1,-0.2,...,0.304405,38,1.469160e+06,252.551253,3.581127,-22.688524,True,True,False,False
1,scale1.00_cap0.55_stress0.15_dd0.06_mom-0.01_b...,wf_2022,1.0,0.55,0.15,-0.06,-0.01,0.55,0.1,-0.2,...,0.078201,20,8.518019e+05,41.768258,1.239695,NaN,True,True,True,True
2,scale1.00_cap0.55_stress0.15_dd0.06_mom-0.01_b...,wf_2023_2024,1.0,0.55,0.15,-0.06,-0.01,0.55,0.1,-0.2,...,0.247830,28,1.336919e+06,4.048510,2.541646,NaN,True,True,True,True


In [6]:
def fold_snapshot(rows: pd.DataFrame) -> list[dict[str, Any]]:
    cols = [
        "fold", "return_pct", "buyhold_return_pct", "relative_return_pp", "max_dd_pct",
        "avg_pos", "trade_count", "relative_gap_pp", "max_dd_gap_pct", "return_floor_gap_pp",
        "relative_gate", "max_dd_gate", "return_floor_gate", "fold_gate",
    ]
    return _clean(rows[cols].to_dict(orient="records"))

failed_folds = [row["fold"] for row in fold_snapshot(best_fold_rows) if not row["fold_gate"]]
wf2020_row = best_fold_rows[best_fold_rows["fold"] == "wf_2020"].iloc[0].to_dict()
wf2022_row = best_fold_rows[best_fold_rows["fold"] == "wf_2022"].iloc[0].to_dict()
wf2324_row = best_fold_rows[best_fold_rows["fold"] == "wf_2023_2024"].iloc[0].to_dict()

if strict_gate_pass_count:
    decision = "hybrid_failure_repair_screen_path_candidate_ready_for_target_replay_no_actor_promotion"
    next_action = "convert the path-level repair into a target contract and rerun three-fold BC/checkpoint replay before actor retention"
else:
    decision = "hybrid_failure_repair_screen_no_three_fold_pass_no_actor_promotion"
    next_action = "do not train actor from the current hybrid repair; build a stronger wf2020 bull/recovery target before rerunning three-fold BC/checkpoint replay"

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "actor_retention_allowed": False,
    "promotion_allowed": False,
    "role": "diagnose_target_repair_screen",
    "changed_layer": ["model_selection", "target_repair", "execution_overlay_diagnostic"],
    "source_card": rel(SOURCE_CARD / "summary.json"),
    "source_decision": source_summary["decision"],
    "source_selected_candidate": source_summary["selected_candidate"],
    "reference_baseline": source_summary["reference_baseline"],
    "canonical_baseline": source_summary["canonical_baseline"],
    "direct_comparison_baseline": source_summary["direct_comparison_baseline"],
    "target_revision_contract": rel(CONTRACT_CARD / "target_revision_contract.json"),
    "target_revision_next_mechanism": contract["next_mechanism"],
    "replay_model": "official_return_scaled_applied_exposure_path_diagnostic",
    "baseline_replay_max_abs_nav_error": float(baseline_check["nav_max_abs_error"].max()),
    "required_wf_windows": FOLDS,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "target_relative_return_by_fold_pp": TARGET_RELATIVE_PP,
    "return_floor_pct_by_fold": RETURN_FLOOR_PCT,
    "grid": GRID,
    "candidate_count": candidate_count,
    "strict_gate_pass_count": strict_gate_pass_count,
    "best_candidate": _clean(best),
    "best_candidate_fold_results": fold_snapshot(best_fold_rows),
    "failed_folds": failed_folds,
    "best_drawdown_safe_candidate": _clean(best_drawdown_safe),
    "best_return_floor_safe_candidate": _clean(best_return_floor_safe),
    "diagnosis": [
        f"Best path-level variant passes {int(best['fold_gate_pass_count'])}/3 fold gates; strict gate pass count is {strict_gate_pass_count}.",
        f"wf_2022 stress repair succeeds for the best variant: return={wf2022_row['return_pct']:.2f}%, relative={wf2022_row['relative_return_pp']:.2f}pp, MaxDD={wf2022_row['max_dd_pct']:.2f}%.",
        f"wf_2023_2024 bull participation repair succeeds for the best variant: return={wf2324_row['return_pct']:.2f}%, relative={wf2324_row['relative_return_pp']:.2f}pp, MaxDD={wf2324_row['max_dd_pct']:.2f}%.",
        f"wf_2020 remains below the {RETURN_FLOOR_PCT['wf_2020']:.2f}% return floor by {abs(wf2020_row['return_floor_gap_pp']):.2f}pp despite high-participation grid variants.",
    ],
    "next_action": next_action,
    "sidecars": {
        "notebook": rel(CARD / f"{EXPERIMENT}.ipynb"),
        "baseline_replay_check": rel(CARD / "baseline_replay_check.csv"),
        "repair_grid_top": rel(CARD / "repair_grid_top.csv"),
        "fold_repair_metrics": rel(CARD / "fold_repair_metrics.csv"),
        "pass_distribution": rel(CARD / "pass_distribution.csv"),
        "summary": rel(CARD / "summary.json"),
        "selection": rel(CARD / "selection.json"),
        "conclusion": rel(CARD / "conclusion.md"),
    },
}

selection = {
    "experiment": EXPERIMENT,
    "selected_variant": best["variant"],
    "selected_variant_gate": bool(best["fold_gate_pass_count"] == len(FOLDS)),
    "selected_variant_metrics": _clean(best),
    "selected_fold_results": fold_snapshot(best_fold_rows),
    "training_allowed_from_this_card": False,
    "actor_retention_allowed": False,
    "next_action": next_action,
}

(CARD / "summary.json").write_text(json.dumps(_clean(summary), indent=2, sort_keys=True) + "\n")
(CARD / "selection.json").write_text(json.dumps(_clean(selection), indent=2, sort_keys=True) + "\n")

conclusion_lines = [
    f"# {EXPERIMENT}",
    "",
    f"Decision: `{decision}`.",
    "",
    f"The best path-level repair is `{best['variant']}` and passes {int(best['fold_gate_pass_count'])}/3 fold gates; no actor promotion is allowed.",
    "",
    f"- wf_2020: return {wf2020_row['return_pct']:.2f}%, relative {wf2020_row['relative_return_pp']:.2f}pp, MaxDD {wf2020_row['max_dd_pct']:.2f}%; still misses the {RETURN_FLOOR_PCT['wf_2020']:.2f}% floor by {abs(wf2020_row['return_floor_gap_pp']):.2f}pp.",
    f"- wf_2022: return {wf2022_row['return_pct']:.2f}%, relative {wf2022_row['relative_return_pp']:.2f}pp, MaxDD {wf2022_row['max_dd_pct']:.2f}%; stress cap repairs the drawdown gate.",
    f"- wf_2023_2024: return {wf2324_row['return_pct']:.2f}%, relative {wf2324_row['relative_return_pp']:.2f}pp, MaxDD {wf2324_row['max_dd_pct']:.2f}%; bull participation repairs the relative-return gate.",
    "",
    f"Next action: {next_action}.",
]
(CARD / "conclusion.md").write_text("\n".join(conclusion_lines) + "\n")

print(json.dumps(_clean({
    "decision": decision,
    "candidate_count": candidate_count,
    "strict_gate_pass_count": strict_gate_pass_count,
    "best_variant": best["variant"],
    "best_fold_gate_pass_count": int(best["fold_gate_pass_count"]),
    "failed_folds": failed_folds,
    "next_action": next_action,
}), indent=2))

{
  "decision": "hybrid_failure_repair_screen_no_three_fold_pass_no_actor_promotion",
  "candidate_count": 302400,
  "strict_gate_pass_count": 0,
  "best_variant": "scale1.00_cap0.55_stress0.15_dd0.06_mom-0.01_bull0.55_mom0.10_dd0.20",
  "best_fold_gate_pass_count": 2,
  "failed_folds": [
    "wf_2020"
  ],
  "next_action": "do not train actor from the current hybrid repair; build a stronger wf2020 bull/recovery target before rerunning three-fold BC/checkpoint replay"
}


## Conclusion

This run does not produce a retained model. The best path-level repair fixes the `wf_2022` drawdown gate and the `wf_2023_2024` relative-return gate, but `wf_2020` still cannot reach the calibrated 69.60% return floor under the screened stress-cap plus bull-participation overlay.

The next runnable step is a target-level change, not UI work and not actor retention: build a stronger `wf_2020` bull/recovery participation target, then rerun three-fold BC/checkpoint replay before considering any retained actor.